# 03 - Reversible training pushed to the MAXIMUM batch size

**Goal (assignment part 3).** Take the winning reversible variant from notebook 02 and see how big a
batch a single GPU can hold now that activations are no longer stored. Train for 50M tokens at that
batch and compare with notebook 01.

Two things to watch for:
1. Reversibility removes the per-layer activations, but the **output logits** (`tokens x vocab`) still
   need memory. With a small model they become the limit. An optional *chunked loss* removes that limit.
2. A giant batch means **few optimizer steps** for the same 50M tokens. We scale the learning rate by
   `sqrt(batch/64)` and also run an un-scaled control so you can see the effect.

In [ ]:
SMOKE = False   # True = tiny CPU version of this whole notebook (used to test the code; not for results)
REPO_URL = "https://github.com/YOUR-USERNAME/reversible-llm-20m"   # <- edit: only needed on Colab
USE_DRIVE = False   # True = keep data + results on Google Drive so they survive a Colab disconnect

import os, sys, copy, json, subprocess
if os.path.exists("../revlm"):
    os.chdir("..")                                   # opened from the notebooks/ folder
elif not os.path.exists("revlm"):
    assert "YOUR-USERNAME" not in REPO_URL, "edit REPO_URL (top of this cell) to point at your GitHub repo"
    subprocess.run(["git", "clone", REPO_URL, "repo"], check=True)   # fresh Colab VM
    os.chdir("repo")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "tokenizers", "datasets"], check=False)

import torch
from IPython.display import Image, display
from revlm import selftest, experiment as ex
from revlm.bench import bench_steps, scaling_table
from revlm.data import prepare_data, Data
from revlm.report import write_summary, plot_scaling

S = ex.settings(SMOKE)
DEV = ex.device()
RESULTS = "results_smoke" if SMOKE else "results"
DATA_DIR = "data_smoke" if SMOKE else "data"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS = "/content/drive/MyDrive/reversible-llm-20m/" + RESULTS
    DATA_DIR = "/content/drive/MyDrive/reversible-llm-20m/" + DATA_DIR
print("device:", DEV, torch.cuda.get_device_name() if DEV == "cuda" else "(no GPU!)" if not SMOKE else "(smoke test)")
if DEV != "cuda" and not SMOKE:
    print("WARNING: no GPU. In Colab use Runtime > Change runtime type > GPU.")

## 0. Sanity check

In [ ]:
# 30-second correctness check on THIS machine/GPU before spending an hour of compute.
assert selftest.main(DEV), "self-test failed - do not trust any numbers below"

## 1. Data

In [ ]:
# Downloads TinyStories, trains an 8k BPE tokenizer, writes exactly 50M training tokens (~3-6 min, cached).
prepare_data(DATA_DIR, vocab_size=8192, **S["data"])
data = Data(DATA_DIR, S["model"].ctx, seed=1337)
print(f"train tokens available: {len(data.train):,}   validation tokens: {len(data.val):,}   sequences: {data.n_seq:,}")

In [ ]:
def mk(mode, **over):
    """20M-parameter model config in the given mode (same weights/shape for every mode)."""
    mc = copy.deepcopy(S["model"]); mc.mode = mode
    for k, v in over.items():
        setattr(mc, k, v)
    return mc

from revlm.model import GPT
print(f"parameters: {GPT(mk('baseline')).num_params()/1e6:.2f}M   (layers={S['model'].n_layer}, d_model={S['model'].d_model}, vocab={S['model'].vocab_size}, ctx={S['model'].ctx})")

In [ ]:
B_FIXED = ex.load(RESULTS, "fixed_batch")["B_fixed"]
win = ex.load(RESULTS, "winner")
WINNER, H = win["winner"], win["h"]
print(f"winner: {WINNER} (h={H}); fixed batch from notebook 01: {B_FIXED}")

## 2. How large a batch fits?

In [ ]:
s_plain = ex.search_or_load("maxbatch_rev_plainloss", mk(WINNER, h=H), RESULTS, S)
s_chunk = ex.search_or_load("maxbatch_rev_chunkedloss", mk(WINNER, h=H, loss_chunk=S["loss_chunk"]), RESULTS, S)
print("max batch, reversible:                ", s_plain["max_batch"])
print("max batch, reversible + chunked loss: ", s_chunk["max_batch"])

## 3. Speed and memory as the batch grows (reversible vs baseline where the baseline still fits)

In [ ]:
ladder = sorted({B_FIXED, *[B_FIXED * k for k in (2, 3, 4, 6, 8) if B_FIXED * k <= s_chunk["max_batch"]],
                 s_plain["max_batch"], s_chunk["max_batch"]})
rows = []
for B in ladder:
    for name, mc in [("baseline", mk("baseline")), (WINNER, mk(WINNER, h=H)),
                     (WINNER + "+chunked_loss", mk(WINNER, h=H, loss_chunk=S["loss_chunk"]))]:
        r = bench_steps(mc, B, DEV, "auto", n_steps=4)
        rows.append(dict(name=name, B=B, **{k: r[k] for k in ("ok", "peak_mib", "tokens_per_sec")}))
        pk = f"{r['peak_mib']:8.0f}" if r["peak_mib"] else "     n/a"
        print(f"B={B:5d} {name:22s}", f"peak {pk} MiB  {r['tokens_per_sec']:9.0f} tok/s" if r["ok"] else "OOM")
ex.save(RESULTS, "batch_ladder", {"rows": rows})

## 4. Train at the maximum batch size

In [ ]:
use_chunk = s_chunk["max_batch"] > s_plain["max_batch"]
B_MAX = max(s_chunk["max_batch"], s_plain["max_batch"])
mc_max = mk(WINNER, h=H, loss_chunk=S["loss_chunk"] if use_chunk else 0)
print(f"training at B={B_MAX}, chunked loss = {use_chunk}")
r3 = ex.run_or_load(mc_max, ex.make_train_cfg(S, "03_rev_maxbatch", B_MAX), data, RESULTS)

Control: the same run with the learning rate NOT scaled up for the bigger batch.

In [ ]:
RUN_LR_CONTROL = True
if RUN_LR_CONTROL:
    r3b = ex.run_or_load(mc_max, ex.make_train_cfg(S, "03_rev_maxbatch_lr_unscaled", r3["batch_size"], lr_scaling="none"), data, RESULTS)

## 5. Final report
`PRICE_PER_HOUR` is **your assumption** for what one hour of this GPU costs on a cloud provider
(Colab free = $0). It only feeds the last column - edit it before quoting cost numbers.

In [ ]:
PRICE_PER_HOUR = 0.35     # assumption - edit
write_summary(RESULTS, price_per_hour=PRICE_PER_HOUR)
display(Image(f"{RESULTS}/loss_curves.png")); display(Image(f"{RESULTS}/speed_memory.png"))
print(open(f"{RESULTS}/summary.md").read())